# Causal Graph Inference

Infer causal structure from interventions: component edge strengths,
information bottlenecks, causal path strength, critical component ordering,
and graph robustness under multi-ablation.

## Why This Matters

Causal graph inference applies rigorous causal inference methods to understanding model internals. Causal methods go beyond correlation to establish which components are genuinely responsible for specific behaviors, providing the strongest form of mechanistic evidence.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.causal_graph_inference import (
    component_causal_edges,
    information_bottleneck_detection,
    causal_path_strength,
    critical_component_ordering,
    graph_robustness,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Component Causal Edges

Estimate causal edge strengths between attention/MLP and the residual stream.

In [ ]:
result = component_causal_edges(model, tokens)
print(f"Strongest edge: {result['strongest_edge']['source']} → {result['strongest_edge']['target']}")
print(f"Total edges: {result['n_edges']}\n")
for e in result['edges']:
    bar = '█' * int(e['strength'] * 100)
    print(f"  {e['source']:12s} → {e['target']:16s}: {e['strength']:.4f} {bar}")

## Information Bottleneck Detection

Find layers where effective dimensionality drops — information bottlenecks.

In [ ]:
result = information_bottleneck_detection(model, tokens)
print(f"Bottleneck layer: {result['bottleneck_layer']} (eff_dim={result['min_effective_dim']:.2f})\n")
for p in result['per_layer']:
    marker = ' ← bottleneck' if p['layer'] == result['bottleneck_layer'] else ''
    print(f"  Layer {p['layer']}: eff_dim={p['effective_dim']:.2f}, norm={p['residual_norm']:.4f}{marker}")

## Causal Path Strength

How much information flows from one layer to another?

In [ ]:
result = causal_path_strength(model, tokens, source_layer=0, target_layer=3)
print(f"Path L0 → L3: disruption={result['path_disruption']:.4f}, "
      f"info_preserved={result['information_preserved']:.4f}\n")
for p in result['per_intermediate']:
    print(f"  Layer {p['layer']}: cosine_to_source={p['cosine_to_source']:.4f}")

## Critical Component Ordering

Rank all components by their causal criticality for final predictions.

In [ ]:
result = critical_component_ordering(model, tokens)
print(f"Most critical: {result['most_critical']['name']}")
print(f"Components for 80% importance: {result['top_80_pct_count']}\n")
for c in result['components'][:10]:
    print(f"  {c['name']:8s}: criticality={c['criticality']:.4f}, "
          f"cumulative={c['cumulative_fraction']:.2%}")

## Graph Robustness

How robust is the model when multiple critical components are ablated?

In [ ]:
result = graph_robustness(model, tokens, n_ablations=3)
print(f"Ablated: {result['ablated_hooks']}")
print(f"Joint effect: {result['joint_effect']:.4f}")
print(f"Sum individual: {result['sum_individual_effects']:.4f}")
print(f"Interaction ratio: {result['interaction_ratio']:.4f}")
print(f"Prediction survival: {result['prediction_survival_rate']:.2%}")
print(f"Robust: {result['is_robust']}")